# Chapter 10: Gluing

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Chapter 10, printed pp. 369-416; PDF pp. 384-431. Sections: 10.1-10.9.

    ## Chapter Goal

    Gluing theorem, connected sums of curves, weighted norms, cutoff functions, construction and derivative of the gluing map, surjectivity, splitting axiom, and a revisited theorem.

    The guiding question is:

    > How can a nodal limit be reversed into nearby smooth curves with controlled error?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| nodal curve | two components joined at a node | matching value |
| neck length | gluing parameter R | error decays like exp(-delta R) |
| cutoff function | smooth interpolation on overlap | derivative support |
| right inverse | linear solve for correction | operator norm bound |
| gluing map | approximate curve plus correction | surjectivity near boundary |


## Standalone Reading Guide

    Gluing is compactness run backward. Compactness says that a sequence of smooth curves may converge to a nodal object. Gluing says that, near a regular nodal object, one can construct nearby smooth curves by cutting ends, inserting a long neck, and correcting the small analytic error. The gluing parameter measures how long the neck is or, equivalently, how close the smoothed curve is to the nodal boundary.

The hard part is not drawing the connected sum; it is proving that the approximate solution can be corrected uniquely and smoothly. Weighted norms make the long neck visible to estimates. Cutoff functions localize the error. A right inverse for the linearized operator feeds a Newton-type argument. Once the gluing map is controlled, the boundary of moduli space can be related to products of lower-dimensional moduli spaces.

The plots below isolate two pieces of the proof. The pregluing error decays exponentially with neck length, while the cutoff derivative lives only in the transition annulus. Together they explain why the correction problem becomes small for large gluing length even though the domain is degenerating.

    ## Topics In This Notebook

    - pregluing by cutting and joining long necks
- weighted norms that see decay along the neck
- cutoff functions and error localization
- Newton correction from approximate to genuine solution
- splitting axiom as a geometric consequence of gluing

    ## Visualization Storyboard

    - A neck-error plot shows exponential improvement as the gluing length grows.
- A cutoff profile marks the transition region where the Cauchy-Riemann defect is created.
- A dependency graph connects estimates, Newton correction, and the splitting axiom.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'chapter-10'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['nodal curve', 'neck length', 'cutoff function', 'right inverse', 'gluing map']
CONCEPT_EDGES = [('nodal curve', 'neck length'), ('neck length', 'cutoff function'), ('cutoff function', 'right inverse'), ('right inverse', 'gluing map')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=16, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: Gluing')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '369-416',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in Gluing. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
R = np.linspace(1, 12, 220)
delta = 0.55
pregluing_error = 2.0 * np.exp(-delta * R)
weighted_error = np.exp(0.18 * R) * pregluing_error
s = np.linspace(-6, 6, 300)
cutoff = 1 / (1 + np.exp(3 * s))
cutoff_derivative = np.gradient(cutoff, s)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), constrained_layout=True)
axes[0].semilogy(R, pregluing_error, label="unweighted error")
axes[0].semilogy(R, weighted_error, label="weighted estimate")
axes[0].set_xlabel("neck length R")
axes[0].set_ylabel("norm")
axes[0].set_title("Exponential pregluing error")
axes[0].legend()
axes[0].grid(True, alpha=0.25)
axes[1].plot(s, cutoff, label="cutoff")
axes[1].plot(s, np.abs(cutoff_derivative), label="|cutoff derivative|")
axes[1].set_title("Transition region for gluing")
axes[1].set_xlabel("neck coordinate")
axes[1].legend()
axes[1].grid(True, alpha=0.25)
fig_path = save_matplotlib(fig, UNIT, "figures", "gluing-neck-error-cutoff.png")
plt.close(fig)

check = {
    "initial_error": float(pregluing_error[0]),
    "final_error": float(pregluing_error[-1]),
    "error_ratio": float(pregluing_error[0] / pregluing_error[-1]),
    "cutoff_derivative_support_fraction": float(np.mean(np.abs(cutoff_derivative) > 0.01)),
    "passed": bool(pregluing_error[-1] < 0.01 * pregluing_error[0] and np.mean(np.abs(cutoff_derivative) > 0.01) < 0.6),
}
check_path = save_json(check, UNIT, "checks", "gluing-error-checks.json")
display_artifact(fig_path, width=860)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'nodal curve', 'computational_object': 'two components joined at a node', 'check': 'matching value'}, {'item': 'neck length', 'computational_object': 'gluing parameter R', 'check': 'error decays like exp(-delta R)'}, {'item': 'cutoff function', 'computational_object': 'smooth interpolation on overlap', 'check': 'derivative support'}, {'item': 'right inverse', 'computational_object': 'linear solve for correction', 'check': 'operator norm bound'}, {'item': 'gluing map', 'computational_object': 'approximate curve plus correction', 'check': 'surjectivity near boundary'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Decrease the decay rate delta or widen the cutoff transition. The notebook will show how slower decay or larger derivative support weakens the error estimate.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - Gluing constructs smooth curves near nodal boundary points.
- Exponential neck estimates are the reason long gluing parameters are manageable.
- The splitting axiom is powered by identifying boundary strata through gluing.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'gluing-neck-error-cutoff.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'gluing-error-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'gluing-error-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
